# 🎙️ VoiceBatch Studio v0.0
यह कोड सीधे आपके Google Drive के `VoiceBatchModels` फोल्डर से मॉडल उठाएगा।

In [ ]:
# @title 🛠️ Step 1: ड्राइव कनेक्ट और लाइब्रेरी सेटअप
import os
from google.colab import drive

print("⏳ लाइब्रेरीज़ इंस्टॉल हो रही हैं...")
!pip install -q coqpit-config coqui-tts gradio librosa soundfile

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

os.makedirs("outputs", exist_ok=True)
print("✅ ड्राइव कनेक्ट हो गई और सेटअप तैयार है!")

In [ ]:
# @title 🚀 Step 2: अनलिमिटेड वॉयस स्टूडियो लॉन्च करें
import gradio as gr
import torch, librosa, re, numpy as np, soundfile as sf
from TTS.api import TTS

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_path = "/content/drive/MyDrive/VoiceBatchModels/"

if os.path.exists(model_path + "model.pth"):
    print("✅ मॉडल मिल गया! इंजन लोड हो रहा है...")
    tts = TTS(model_path=model_path, config_path=model_path + "config.json").to(device)
    print("🚀 इंजन तैयार है!")
else:
    print("❌ एरर: ड्राइव में मॉडल नहीं मिला।")

def voice_gen(text, audio_sample):
    parts = re.split(r'(?<=[।?!])\s+', text)
    combined = []
    for p in parts:
        if len(p.strip()) < 2: continue
        tts.tts_to_file(text=p, speaker_wav=audio_sample, language='hi', file_path='temp.wav')
        y, _ = librosa.load('temp.wav', sr=24000)
        combined.extend(y)
    
    output_path = "outputs/VoiceBatch_Result.wav"
    sf.write(output_path, np.array(combined), 24000)
    return output_path

gr.Interface(fn=voice_gen, 
             inputs=[gr.Textbox(label="Script", lines=10), 
                     gr.Audio(label="Sample", type='filepath')],
             outputs=gr.Audio()).launch(share=True)